In [ ]:
# Score financial news headlines using a free LLM API, mimicking FinBERT's output.
#
# Set your Groq key in the environment (NEVER hard-code it):
#     export GROQ_API_KEY="gsk_..."   # get one at https://console.groq.com/keys
#
# Install:
#     pip install requests
#
# This script asks an LLM to produce the same shape of output as FinBERT,
# plus a single compound score in [-1, 1].

import os
import json
import time
import requests
from typing import Optional

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError("Set GROQ_API_KEY in your environment first.")

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
GROQ_MODEL = "openai/gpt-oss-20b"  # fast + free-tier friendly

SYSTEM_PROMPT = """You are a financial sentiment classifier that mimics FinBERT.
Given a financial news headline, classify its sentiment from the perspective
of a company/stock/market impact (not general tone).

Respond with ONLY a JSON object, no prose, no markdown fences, in this exact form:
{"positive": <float>, "negative": <float>, "neutral": <float>, "label": "<positive|negative|neutral>"}

Rules:
- positive + negative + neutral must sum to 1.0 (each between 0 and 1)
- "label" is whichever of the three has the highest probability
- Base the judgement on likely market/financial impact, e.g.:
  - "Company X beats earnings estimates" -> positive
  - "Company X faces regulatory investigation" -> negative
  - "Company X to report earnings next Tuesday" -> neutral
"""


def _call_llm(headline: str, retries: int = 3, timeout: int = 20) -> Optional[dict]:
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": GROQ_MODEL,
        "temperature": 0,
        "max_tokens": 200,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Headline: {headline}"},
        ],
    }

    last_err = None
    last_raw = None
    for attempt in range(retries):
        try:
            resp = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=timeout)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"].strip()
            last_raw = content

            if "```" in content:
                content = content.split("```")[1] if content.count("```") >= 2 else content
                content = content.replace("json", "", 1).strip()

            start = content.find("{")
            end = content.rfind("}")
            if start == -1 or end == -1 or end < start:
                raise json.JSONDecodeError("No JSON object found in response", content, 0)
            content = content[start:end + 1]

            return json.loads(content)
        except (requests.RequestException, json.JSONDecodeError, KeyError, IndexError) as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))

    raise RuntimeError(
        f"Failed to get a valid response after {retries} attempts: {last_err}\n"
        f"Last raw model output: {last_raw!r}"
    )


def score_headline(headline: str) -> dict:
    """
    Returns a FinBERT-style sentiment dict plus a compound score:
        {
            "headline": str,
            "positive": float,
            "negative": float,
            "neutral": float,
            "label": "positive" | "negative" | "neutral",
            "compound_score": float in [-1, 1]
        }
    """
    result = _call_llm(headline)

    pos = float(result.get("positive", 0))
    neg = float(result.get("negative", 0))
    neu = float(result.get("neutral", 0))

    total = pos + neg + neu
    if total > 0:
        pos, neg, neu = pos / total, neg / total, neu / total

    label = result.get("label") or max(
        [("positive", pos), ("negative", neg), ("neutral", neu)], key=lambda x: x[1]
    )[0]

    compound_score = pos - neg

    return {
        "headline": headline,
        "positive": round(pos, 4),
        "negative": round(neg, 4),
        "neutral": round(neu, 4),
        "label": label,
        "compound_score": round(compound_score, 4),
    }


def score_headlines(headlines: list, delay: float = 0.0) -> list:
    """Batch helper. `delay` (seconds) helps stay under free-tier rate limits."""
    results = []
    for h in headlines:
        results.append(score_headline(h))
        if delay:
            time.sleep(delay)
    return results


if __name__ == "__main__":
    sample_headlines = [
        "Apple beats Q3 earnings estimates, raises full-year guidance",
        "Tesla shares plunge after CEO announces surprise resignation",
        "Federal Reserve to hold interest rate meeting next Wednesday",
        "Amazon faces antitrust lawsuit over marketplace practices",
    ]

    for headline in sample_headlines:
        result = score_headline(headline)
        print(json.dumps(result, indent=2))
